<a href="https://colab.research.google.com/github/ricardoproenca/AI/blob/main/RAG_LOCAL_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chatbot pdf colab · PY

# ANTES DE RODAR:
#   Runtime → Change runtime type → T4 GPU  ✅

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
    --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 848.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00


In [2]:
# ========================
# CÉLULA 2 — Fazer upload do PDF
# ========================
from google.colab import files

print("📄 Selecione seu PDF:")
uploaded = files.upload()

caminho_pdf = list(uploaded.keys())[0]
print(f"✅ PDF carregado: {caminho_pdf}")

📄 Selecione seu PDF:


KeyboardInterrupt: 

In [ ]:
from google.colab import files
uploaded = files.upload()

# Coloque exatamente o nome do arquivo
loader = PyPDFLoader("/content/Manual_Basico_do_Windows_11_v1.pdf")
docs = loader.load()

In [ ]:
# ========================
# CÉLULA 3 — Baixar modelo GGUF do Hugging Face
# ========================
# Modelo recomendado: Mistral 7B Q4_K_M (~4 GB) — melhor que WizardLM para português
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download

print("⏳ Baixando modelo (pode demorar ~5 min dependendo da conexão)...")

caminho_modelo = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    local_dir="/content/models"
)

print(f"✅ Modelo salvo em: {caminho_modelo}")

In [ ]:
# ========================
# CÉLULA 4 — Código principal
# ========================
import numpy as np
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from llama_cpp import Llama

# ---------- Configurações ----------
CHUNK_TAMANHO = 300
CHUNK_OVERLAP = 50
TOP_K         = 3
DIST_MAX      = 1.5
MAX_TOKENS    = 300
N_CTX         = 4096

# ---------- 1. Ler PDF ----------
def extrair_texto_pdf(caminho: str) -> str:
    reader = PdfReader(caminho)
    texto = ""
    for page in reader.pages:
        t = page.extract_text()
        if t:
            texto += t + "\n"
    if not texto.strip():
        raise ValueError("Nenhum texto extraído. O PDF pode ser uma imagem escaneada.")
    print(f"✅ PDF lido — {len(reader.pages)} páginas, {len(texto.split())} palavras")
    return texto

# ---------- 2. Chunks com sobreposição ----------
def chunk_text(texto: str) -> list:
    palavras = texto.split()
    chunks, i = [], 0
    while i < len(palavras):
        chunks.append(" ".join(palavras[i:i + CHUNK_TAMANHO]))
        i += CHUNK_TAMANHO - CHUNK_OVERLAP
    print(f"✅ {len(chunks)} chunks criados")
    return chunks

# ---------- 3. Embeddings ----------
def criar_embeddings(chunks: list):
    print("⏳ Gerando embeddings...")
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
    id2chunk = {i: chunk for i, chunk in enumerate(chunks)}
    print("✅ Embeddings prontos")
    return embedder, embeddings, id2chunk

# ---------- 4. Índice FAISS ----------
def criar_index(embeddings: np.ndarray):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    print(f"✅ Índice FAISS criado com {index.ntotal} vetores")
    return index

# ---------- 5. Recuperar contexto ----------
def recuperar_contexto(pergunta, embedder, index, id2chunk):
    query_vec = embedder.encode([pergunta], convert_to_numpy=True)
    D, I = index.search(query_vec, TOP_K)
    resultados = [
        id2chunk[i] for i, d in zip(I[0], D[0])
        if i in id2chunk and d < DIST_MAX
    ]
    if not resultados:
        return None
    return "\n\n---\n\n".join(resultados)

# ---------- 6. Carregar modelo ----------
def carregar_modelo(caminho_gguf: str):
    print("⏳ Carregando modelo na GPU...")
    model = Llama(
        model_path=caminho_gguf,
        n_ctx=N_CTX,
        n_gpu_layers=-1,   # -1 = todas as camadas na GPU (máximo desempenho)
        verbose=False
    )
    print("✅ Modelo carregado na GPU!")
    return model

# ---------- 7. Gerar resposta ----------
def perguntar(pergunta, model, embedder, index, id2chunk):
    contexto = recuperar_contexto(pergunta, embedder, index, id2chunk)

    if contexto is None:
        return "Não encontrei nenhum trecho relevante no documento para essa pergunta."

    prompt = f"""<s>[INST] Você é um assistente prestativo. Responda SEMPRE em português do Brasil.
Use APENAS o contexto abaixo. Se a resposta não estiver no contexto, diga: "Não encontrei essa informação no documento."

Contexto:
{contexto}

Pergunta: {pergunta} [/INST]"""

    output = model(
        prompt,
        max_tokens=MAX_TOKENS,
        stop=["</s>", "[INST]"],
        echo=False,
        temperature=0.2
    )
    return output["choices"][0]["text"].strip()

# ---------- 8. Inicializar tudo ----------
print("\n🚀 Inicializando chatbot...\n")

texto    = extrair_texto_pdf(caminho_pdf)
chunks   = chunk_text(texto)
embedder, embeddings, id2chunk = criar_embeddings(chunks)
index    = criar_index(embeddings)
model    = carregar_modelo(caminho_modelo)

print("\n" + "="*50)
print("💬 Chatbot pronto! Execute a célula abaixo para conversar.")
print("="*50)

In [ ]:
# ========================
# CÉLULA 5 — Chat interativo
# ========================
# Execute esta célula quantas vezes quiser para fazer perguntas

pergunta = input("Você: ")
resposta = perguntar(pergunta, model, embedder, index, id2chunk)
print(f"\nChatbot: {resposta}")